In [1]:
import sys
import os

# Get the parent directory (project root)
# this assumes notebooks is a dir within root dir
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)



In [2]:
### on myriad, the prefix of this dir means it can't be used in other file paths
### i don't know what that means
### but don't get confused with other file paths set below
print(project_root)

## for reloading functions before running. helps when developing code
%load_ext autoreload
%autoreload 2

/myriadfs/home/ucsagil/Scratch/image-analysis/cellpose-segmentation-demo


In [3]:
from scripts.load_images import load_images
from scripts.segment import save_mask_overlay, segment_frames, estimate_diameter, validate_segmentation_objects
from scripts.track import extract_centroids, track_particles
from scripts.utils import plot_size_distribution

import yaml




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [9]:
config_file = '/home/ucsagil/Scratch/image-analysis/cellpose-segmentation-demo/configs/params.yaml'

with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)
    print('config loaded...')

#### so you know what's happening
testing = cfg['testing']
frame_subset = cfg['frame_subset']
subset_frame_number = cfg['subset_frame_number']
outdir = cfg['outdir']
print('Some things for you to check before proceeding:')
print('testing is set to ' + testing)
print('if testing is true, we will downsize images to ' + downsize_size + ' to improve performance')
print('using a subset of frames is ' + frame_subset)
print( 'if using a subset of frames is true, we will use ' + str(subset_frame_number) + ' frames' )

config loaded...


In [10]:
frames_in, tif_files = load_images(cfg['folder'], testing=testing)
if frames_in.any():
    print('***images loaded***')
else:
    print('***warning: no images loaded***')
    
# may only want to run on a subset of frames
if frame_subset:
    print("subsetting to use " + str(cfg['subset_frame_number']) + " frames")
    frames = frames_in[:cfg['subset_frame_number']]
else:
    print( "using all " + str(len(frames_in)) + " frames")
    frames = frames_in

Found 92 TIFF files.
Downsizing frames to (100, 100) for testing...
***images loaded***
using all 92 frames


In [ ]:
masks = segment_frames(
    frames,
    gpu=cfg['use_gpu'],
    output_dir=outdir,
    save_overlays=True,
    file_names=[os.path.basename(f) for f in tif_files]  # optional
)

segmenting image t000.tif
saving mask for image t000.tif
segmenting image t001.tif
saving mask for image t001.tif
segmenting image t002.tif
saving mask for image t002.tif
segmenting image t003.tif
saving mask for image t003.tif
segmenting image t004.tif
saving mask for image t004.tif
segmenting image t005.tif
saving mask for image t005.tif
segmenting image t006.tif
saving mask for image t006.tif
segmenting image t007.tif
saving mask for image t007.tif
segmenting image t008.tif
saving mask for image t008.tif
segmenting image t009.tif
saving mask for image t009.tif
segmenting image t010.tif
saving mask for image t010.tif
segmenting image t011.tif
saving mask for image t011.tif
segmenting image t012.tif
saving mask for image t012.tif
segmenting image t013.tif
saving mask for image t013.tif
segmenting image t014.tif
saving mask for image t014.tif
segmenting image t015.tif
saving mask for image t015.tif
segmenting image t016.tif
saving mask for image t016.tif
segmenting image t017.tif
savin

In [ ]:
##### some stuff about validating

# 1. check boundaries

# 2. check masks

# 3. check diameters

# 4. size histograms - are there any outliers? Are these mistakes?





In [ ]:

### first let's look at some frames in more detail.
validate_segmentation_objects(
    frames,
    masks,
    file_names=tif_files,
    output_dir=cfg['output_dir'],
    sample_fraction=cfg.get('validate_sample_fraction', 0.05),
    min_samples=cfg.get('validate_min_samples', 5)
)



In [ ]:
### Now we are confident (or not :'D )the segmentation and size estimates are generally correct
### let's look at the overall distribution

## we plot the distribution of all objects
## small outliers might suggest small fragments have been classified as objects
## large outliers might indicate poor separation of close objects
plot_size_distribution(
    masks=masks,
    file_names=tif_files,
    output_dir=cfg['output_dir']
)